# Chapter 7: Uncertainty Propagation

<a href="../lite/lab/index.html?path=ch07_uncertainty_propagation.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import scipy.stats as stats

plt.rcParams.update({
    'figure.figsize': (9, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'lines.linewidth': 1.5,
    'figure.dpi': 100,
})


def draw_covariance_ellipse(ax, mean, cov, n_std=2, **kwargs):
    """Draw a covariance ellipse centered at `mean`.

    Parameters
    ----------
    ax : matplotlib Axes
    mean : array-like, shape (2,)
    cov : array-like, shape (2, 2)
    n_std : float, number of standard deviations for the ellipse radius
    **kwargs : forwarded to matplotlib Ellipse
    """
    vals, vecs = np.linalg.eigh(cov)
    # Clamp tiny negative eigenvalues that arise from numerical noise
    vals = np.maximum(vals, 0.0)
    angle = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
    w, h = 2 * n_std * np.sqrt(vals)
    ellipse = Ellipse(xy=mean, width=w, height=h, angle=angle, **kwargs)
    ax.add_patch(ellipse)
    return ellipse


print("Imports and helpers ready.")

Your robot drives in a circle. Its odometry says it returned exactly to the start.
But the **uncertainty ellipse** says it could be anywhere in a 3 meter blob.
Where did the certainty go?

This chapter explains how uncertainty grows, how it **propagates** through
nonlinear functions, and why the **Jacobian** is the single most important tool
for managing it.

```{admonition} What you will build
:class: tip

- Propagate a robot's position uncertainty through a nonlinear motion model using the Jacobian
- Watch covariance ellipses grow as a robot drives without measurements (dead reckoning)
- See when linearization breaks down and produces the famous banana shaped distribution
- Compare Jacobian based propagation with Monte Carlo sampling

**Real world application:** Understanding uncertainty growth is critical for knowing when your robot's estimate is trustworthy and when it needs a sensor update. This is the foundation of EKF prediction.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **JAX / autograd** | Automatic differentiation for computing Jacobians without manual derivation |
| **numdifftools** | Numerical Jacobian computation for verification |
| **Ceres Solver** | Google's C++ library for automatic Jacobian computation via autodiff |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

---
## 7.1 Noise in Motion and Sensing

Every actuator adds noise: motors have backlash, wheels slip, and gearboxes
introduce friction that varies with temperature. Every sensor reading is noisy
too: wheel encoders suffer from quantization, LiDAR beams scatter off
reflective surfaces, and cameras blur in low light.

Even if each individual error is tiny, these errors **accumulate** over time.
A robot driving in a straight line will slowly drift sideways; a robot turning
will accumulate heading errors that corrupt all future positions.

Let's see this in action. We will simulate a robot that tries to drive straight
forward, but its velocity command is corrupted by Gaussian noise at every
time step.

In [ ]:
# ── PARAMETERS ── change these and re-run ────
true_velocity = 1.0        # m/s commanded velocity
sigma_v       = 0.1        # std dev of velocity noise
sigma_heading = 0.02       # std dev of heading noise per step (rad)
n_steps       = 50         # number of time steps
n_trials      = 100        # number of Monte Carlo trials
dt            = 0.1        # time step (s)
# ─────────────────────────────────────────────

np.random.seed(42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Store endpoints
endpoints = np.zeros((n_trials, 2))

for trial in range(n_trials):
    xs, ys = [0.0], [0.0]
    theta = 0.0
    for _ in range(n_steps):
        v = true_velocity + np.random.randn() * sigma_v
        theta += np.random.randn() * sigma_heading
        xs.append(xs[-1] + v * np.cos(theta) * dt)
        ys.append(ys[-1] + v * np.sin(theta) * dt)
    axes[0].plot(xs, ys, color='steelblue', alpha=0.15, linewidth=0.8)
    endpoints[trial] = [xs[-1], ys[-1]]

# Ideal trajectory
ideal_x = true_velocity * dt * n_steps
axes[0].plot([0, ideal_x], [0, 0], 'k--', linewidth=2, label='ideal path')
axes[0].plot(0, 0, 'ko', markersize=8)
axes[0].set_xlabel('x (m)')
axes[0].set_ylabel('y (m)')
axes[0].set_title(f'{n_trials} Noisy Trajectories')
axes[0].legend()
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# Endpoint scatter
axes[1].scatter(endpoints[:, 0], endpoints[:, 1], color='tomato', s=15, alpha=0.6)
mean_ep = endpoints.mean(axis=0)
cov_ep = np.cov(endpoints.T)
draw_covariance_ellipse(axes[1], mean_ep, cov_ep, n_std=2,
                        fill=False, edgecolor='tomato', linewidth=2, label='2$\\sigma$ ellipse')
axes[1].plot(ideal_x, 0, 'k*', markersize=12, label='ideal endpoint')
axes[1].set_xlabel('x (m)')
axes[1].set_ylabel('y (m)')
axes[1].set_title('Endpoint Spread')
axes[1].legend()
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key Observations:**

1. Each individual trajectory looks reasonable, yet the **cloud of endpoints**
   is surprisingly large.
2. Heading noise is especially destructive: a small angular error early on
   deflects the entire remaining path.
3. Uncertainty grows with every step. After $n$ steps the position spread
   is roughly proportional to $\sqrt{n}$ for velocity noise, but can grow
   **faster** when heading noise is present.
4. Try increasing `sigma_heading` to see how quickly the cloud explodes.

---
## 7.2 Linearization via Jacobians

We need a systematic way to predict how uncertainty transforms when it passes
through a function. The key idea is **linearization**: approximate a nonlinear
function by its first order Taylor expansion around the current estimate.

For a function $\mathbf{y} = f(\mathbf{x})$, the **Jacobian** matrix $J$
contains all first order partial derivatives:

$$J = \frac{\partial f}{\partial \mathbf{x}}\bigg|_{\mathbf{x}_0}
= \begin{bmatrix}
\frac{\partial f_1}{\partial x_1} & \cdots & \frac{\partial f_1}{\partial x_n} \\
\vdots & \ddots & \vdots \\
\frac{\partial f_m}{\partial x_1} & \cdots & \frac{\partial f_m}{\partial x_n}
\end{bmatrix}$$

The Jacobian tells us: if you wiggle $\mathbf{x}$ by a tiny amount
$\delta\mathbf{x}$, the output changes by approximately
$\delta\mathbf{y} \approx J \, \delta\mathbf{x}$.

Let's see this with a concrete example: converting **polar coordinates**
(range $r$, bearing $\theta$) to **Cartesian coordinates** $(x, y)$.

$$f(r, \theta) = \begin{bmatrix} r \cos\theta \\ r \sin\theta \end{bmatrix}$$

The Jacobian is:

$$J = \begin{bmatrix}
\cos\theta & -r\sin\theta \\
\sin\theta & r\cos\theta
\end{bmatrix}$$

In [ ]:
# ── PARAMETERS ── change these and re-run ────
r0          = 5.0      # nominal range (m)
theta0_deg  = 45.0     # nominal bearing (degrees)
sigma_r     = 0.3      # std dev of range noise (m)
sigma_theta = 0.08     # std dev of bearing noise (rad)
n_samples   = 1000     # Monte Carlo samples
# ─────────────────────────────────────────────

theta0 = np.radians(theta0_deg)

# Covariance in polar space
Sigma_polar = np.diag([sigma_r**2, sigma_theta**2])

# Jacobian of polar-to-Cartesian at (r0, theta0)
J = np.array([
    [np.cos(theta0), -r0 * np.sin(theta0)],
    [np.sin(theta0),  r0 * np.cos(theta0)]
])

# Linearized covariance in Cartesian space
Sigma_cart_lin = J @ Sigma_polar @ J.T

# Nominal Cartesian point
x0 = r0 * np.cos(theta0)
y0 = r0 * np.sin(theta0)

# Monte Carlo: sample in polar, convert through true nonlinear function
np.random.seed(7)
polar_samples = np.random.multivariate_normal([r0, theta0], Sigma_polar, n_samples)
cart_true = np.column_stack([
    polar_samples[:, 0] * np.cos(polar_samples[:, 1]),
    polar_samples[:, 0] * np.sin(polar_samples[:, 1])
])

# Linearized mapping: delta_cart = J @ delta_polar
delta_polar = polar_samples - np.array([r0, theta0])
cart_lin = np.array([x0, y0]) + (J @ delta_polar.T).T

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, pts, title, color in [
    (axes[0], cart_true, 'True Nonlinear Mapping', 'steelblue'),
    (axes[1], cart_lin, 'Linearized (Jacobian) Mapping', 'orange')
]:
    ax.scatter(pts[:, 0], pts[:, 1], s=3, alpha=0.3, color=color)
    ax.plot(x0, y0, 'k+', markersize=12, markeredgewidth=2)
    draw_covariance_ellipse(ax, [x0, y0], Sigma_cart_lin, n_std=2,
                            fill=False, edgecolor='tomato', linewidth=2,
                            linestyle='--', label='Linearized $2\\sigma$')
    # Also show the Monte Carlo covariance for comparison
    cov_mc = np.cov(cart_true.T)
    draw_covariance_ellipse(ax, cart_true.mean(axis=0), cov_mc, n_std=2,
                            fill=False, edgecolor='forestgreen', linewidth=2,
                            label='Monte Carlo $2\\sigma$')
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')
    ax.set_title(title)
    ax.legend(fontsize=10)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key Observations:**

1. The **true** (nonlinear) mapping produces a slightly curved cloud of points.
   The **linearized** mapping produces a straight elliptical cloud.
2. For moderate noise levels, the two are nearly identical. The linearized
   covariance ellipse (red dashed) and the Monte Carlo ellipse (green) overlap
   closely.
3. Try increasing `sigma_theta` to 0.3 or more. You will see the true
   distribution become banana shaped while the linearized version remains an
   ellipse.
4. The Jacobian captures how the function **stretches and rotates** uncertainty.
   Range noise maps mostly along the radial direction; bearing noise maps
   along the tangential direction.

---
## 7.3 First Order Error Propagation

Now we have the central formula of this chapter. Given a random variable
$\mathbf{x}$ with **covariance** $\Sigma_x$ and a function
$\mathbf{y} = f(\mathbf{x})$, the propagated covariance is:

$$\boxed{\Sigma_y = J \, \Sigma_x \, J^T}$$

Why does this work? Recall that covariance measures how variables vary
together. The Jacobian $J$ is a linear map that tells us how small
perturbations in $\mathbf{x}$ produce perturbations in $\mathbf{y}$. Since
$\delta\mathbf{y} \approx J\,\delta\mathbf{x}$, we have:

$$\Sigma_y = E[\delta\mathbf{y}\,\delta\mathbf{y}^T]
= E[J\,\delta\mathbf{x}\,(J\,\delta\mathbf{x})^T]
= J\,E[\delta\mathbf{x}\,\delta\mathbf{x}^T]\,J^T
= J\,\Sigma_x\,J^T$$

The Jacobian **stretches**, **rotates**, and **scales** the covariance
ellipsoid. Directions in which the function is sensitive (large partial
derivatives) get amplified; directions where it is insensitive get compressed.

Let's apply this to a 2D robot **motion model**. The robot is at pose
$(x, y, \theta)$ and executes a motion command: drive forward by distance $d$
and turn by $\delta\theta$. The motion model is:

$$f(x, y, \theta, d, \delta\theta) = \begin{bmatrix}
x + d\cos(\theta + \delta\theta) \\
y + d\sin(\theta + \delta\theta) \\
\theta + \delta\theta
\end{bmatrix}$$

In [ ]:
# ── PARAMETERS ── change these and re-run ────
x0, y0, theta0_mot = 0.0, 0.0, np.radians(30)  # initial pose
d            = 2.0       # forward distance (m)
delta_theta  = np.radians(20)  # turning angle (rad)
sigma_d      = 0.15      # std dev of distance noise
sigma_dtheta = 0.05      # std dev of turning noise (rad)
n_mc         = 1000      # Monte Carlo samples
# ─────────────────────────────────────────────

def motion_model(pose, ctrl):
    """Apply motion command to pose."""
    x, y, th = pose
    dd, dth = ctrl
    return np.array([
        x + dd * np.cos(th + dth),
        y + dd * np.sin(th + dth),
        th + dth
    ])

def motion_jacobian_state(pose, ctrl):
    """Jacobian of motion model w.r.t. pose (3x3)."""
    x, y, th = pose
    dd, dth = ctrl
    return np.array([
        [1, 0, -dd * np.sin(th + dth)],
        [0, 1,  dd * np.cos(th + dth)],
        [0, 0,  1]
    ])

def motion_jacobian_ctrl(pose, ctrl):
    """Jacobian of motion model w.r.t. control (3x2)."""
    x, y, th = pose
    dd, dth = ctrl
    return np.array([
        [np.cos(th + dth), -dd * np.sin(th + dth)],
        [np.sin(th + dth),  dd * np.cos(th + dth)],
        [0,                 1]
    ])

pose0 = np.array([x0, y0, theta0_mot])
ctrl0 = np.array([d, delta_theta])

# Noise covariance on control
Q_ctrl = np.diag([sigma_d**2, sigma_dtheta**2])

# Propagated pose
pose1 = motion_model(pose0, ctrl0)

# Jacobians
Fx = motion_jacobian_state(pose0, ctrl0)
Fu = motion_jacobian_ctrl(pose0, ctrl0)

# Initial covariance (assume we know the start perfectly)
Sigma0 = np.zeros((3, 3))

# Propagated covariance:  Sigma1 = Fx @ Sigma0 @ Fx.T + Fu @ Q_ctrl @ Fu.T
Sigma1 = Fx @ Sigma0 @ Fx.T + Fu @ Q_ctrl @ Fu.T

# Monte Carlo comparison
np.random.seed(42)
ctrl_samples = np.random.multivariate_normal(ctrl0, Q_ctrl, n_mc)
mc_poses = np.array([motion_model(pose0, c) for c in ctrl_samples])

fig, ax = plt.subplots(figsize=(9, 7))

# Monte Carlo samples (xy only)
ax.scatter(mc_poses[:, 0], mc_poses[:, 1], s=4, alpha=0.3,
           color='steelblue', label='Monte Carlo samples')

# Linearized covariance ellipse (xy sub-block)
draw_covariance_ellipse(ax, pose1[:2], Sigma1[:2, :2], n_std=2,
                        fill=False, edgecolor='tomato', linewidth=2.5,
                        label='Linearized $2\\sigma$')

# Monte Carlo covariance ellipse
mc_cov = np.cov(mc_poses[:, :2].T)
draw_covariance_ellipse(ax, mc_poses[:, :2].mean(axis=0), mc_cov, n_std=2,
                        fill=False, edgecolor='forestgreen', linewidth=2,
                        linestyle='--', label='Monte Carlo $2\\sigma$')

# Draw start and end poses
ax.plot(x0, y0, 'ko', markersize=10, label='start')
arrow_len = 0.4
ax.annotate('', xy=(x0 + arrow_len*np.cos(theta0_mot),
                     y0 + arrow_len*np.sin(theta0_mot)),
            xytext=(x0, y0),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

ax.plot(pose1[0], pose1[1], 'k^', markersize=10, label='predicted pose')
ax.annotate('', xy=(pose1[0] + arrow_len*np.cos(pose1[2]),
                     pose1[1] + arrow_len*np.sin(pose1[2])),
            xytext=(pose1[0], pose1[1]),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('Motion Model: Linearized vs Monte Carlo Propagation')
ax.legend(loc='upper left')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Propagated covariance (3x3):")
print(np.round(Sigma1, 6))

**Key Observations:**

1. The formula $\Sigma_y = J\,\Sigma_x\,J^T$ produces a covariance ellipse
   (red) that matches the Monte Carlo scatter (green dashed) almost perfectly.
2. The ellipse is **elongated** in the direction perpendicular to the heading.
   That is because turning noise rotates the entire forward motion vector,
   producing a large lateral error.
3. The Jacobian has two parts: $F_x$ (how pose uncertainty propagates) and
   $F_u$ (how control noise enters). The full propagation formula is
   $\Sigma_1 = F_x\,\Sigma_0\,F_x^T + F_u\,Q\,F_u^T$.
4. Try increasing `sigma_dtheta` to 0.2 and watch how the ellipse stretches
   perpendicular to the heading direction.

---
## 7.4 Covariance Growth over a Trajectory

In practice, a robot does not execute a single motion. It drives for hundreds
or thousands of steps. At each step the prediction formula applies:

$$\Sigma_{k+1} = F_x\,\Sigma_k\,F_x^T + F_u\,Q\,F_u^T$$

Because $Q$ (the process noise) is added at every step, the covariance
**grows monotonically**. This is the fundamental problem of **dead reckoning**:
without measurements, uncertainty never decreases.

Let's watch a robot drive along a curved path and observe its covariance
ellipses growing at each step.

In [ ]:
# ── PARAMETERS ── change these and re-run ────
n_steps_traj = 50        # number of steps
v_traj       = 0.5       # forward velocity per step (m)
omega_traj   = 0.05      # turning rate per step (rad)
sigma_v_traj = 0.05      # velocity noise std
sigma_w_traj = 0.02      # turning noise std
ellipse_interval = 5     # draw ellipse every N steps
# ─────────────────────────────────────────────

# Storage
poses = np.zeros((n_steps_traj + 1, 3))
poses[0] = [0.0, 0.0, 0.0]
Sigma = np.zeros((3, 3))  # start with zero uncertainty
sigmas = [Sigma.copy()]
Q_traj = np.diag([sigma_v_traj**2, sigma_w_traj**2])
traces = [0.0]

for k in range(n_steps_traj):
    ctrl = np.array([v_traj, omega_traj])
    Fx = motion_jacobian_state(poses[k], ctrl)
    Fu = motion_jacobian_ctrl(poses[k], ctrl)
    Sigma = Fx @ Sigma @ Fx.T + Fu @ Q_traj @ Fu.T
    poses[k+1] = motion_model(poses[k], ctrl)
    sigmas.append(Sigma.copy())
    traces.append(np.trace(Sigma[:2, :2]))  # trace of xy covariance

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: trajectory with covariance ellipses
ax = axes[0]
ax.plot(poses[:, 0], poses[:, 1], 'k-', linewidth=1.5, label='trajectory')
ax.plot(poses[0, 0], poses[0, 1], 'ko', markersize=8)

for k in range(0, n_steps_traj + 1, ellipse_interval):
    # Color gradient from steelblue to tomato
    t = k / n_steps_traj
    color = plt.cm.coolwarm(t)
    if np.trace(sigmas[k][:2, :2]) > 1e-10:
        draw_covariance_ellipse(ax, poses[k, :2], sigmas[k][:2, :2],
                                n_std=2, fill=True, alpha=0.25,
                                facecolor=color, edgecolor=color,
                                linewidth=1.5)
    ax.plot(poses[k, 0], poses[k, 1], 'o', color=color, markersize=5)

ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('Covariance Ellipses Along Trajectory')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Right: covariance trace over time
ax2 = axes[1]
ax2.plot(range(n_steps_traj + 1), traces, color='tomato', linewidth=2)
ax2.set_xlabel('Time step')
ax2.set_ylabel('tr($\\Sigma_{xy}$) (m$^2$)')
ax2.set_title('Total Position Uncertainty Over Time')
ax2.grid(True, alpha=0.3)
ax2.fill_between(range(n_steps_traj + 1), traces, alpha=0.15, color='tomato')

plt.tight_layout()
plt.show()

**Key Observations:**

1. The covariance ellipses grow **relentlessly**. The robot becomes less and
   less certain about where it is with every step.
2. The trace of the covariance (total variance) grows roughly **linearly**
   in the number of steps for this simple model. With heading noise, it can
   grow **quadratically** or faster.
3. This is precisely why dead reckoning alone is hopeless for long trajectories.
   After 50 steps the uncertainty blob may be meters wide.
4. We **need measurements** (landmarks, GPS, loop closures) to reduce
   uncertainty. That is the subject of future chapters on filtering and SLAM.

---
## 7.5 Limits of Linearization

The first order approximation $\Sigma_y = J\,\Sigma_x\,J^T$ works beautifully
when the noise is small relative to the curvature of the function. But what
happens when the noise is **large**?

Consider a robot that turns by a large angle. If the turning noise is large,
the true distribution of endpoints forms a **banana** (an arc), not an
ellipse. The linearized approximation completely misses this shape.

This is a fundamental limitation of the **Extended Kalman Filter** (EKF),
which relies on first order linearization. When it breaks down, we turn to:

- The **Unscented Kalman Filter** (UKF), which uses carefully chosen sigma
  points to capture the curvature of the function.
- **Particle filters**, which represent the distribution with a cloud of
  weighted samples and handle arbitrary nonlinearity.

In [ ]:
# ── PARAMETERS ── change these and re-run ────
turn_angle_deg  = 90     # nominal turn angle (degrees)
sigma_angle_deg = 30     # std dev of angle noise (try 5, 15, 30, 45)
drive_dist      = 3.0    # forward distance after turn (m)
n_banana        = 3000   # number of Monte Carlo samples
# ─────────────────────────────────────────────

turn_angle = np.radians(turn_angle_deg)
sigma_angle = np.radians(sigma_angle_deg)

def turn_and_drive(theta_noise):
    """Robot at origin facing right turns by (turn_angle + noise) then drives forward."""
    th = turn_angle + theta_noise
    return np.array([drive_dist * np.cos(th),
                     drive_dist * np.sin(th)])

# Nominal endpoint
nominal = turn_and_drive(0.0)

# Jacobian of turn_and_drive w.r.t. theta_noise at theta_noise=0
J_td = np.array([[-drive_dist * np.sin(turn_angle)],
                  [ drive_dist * np.cos(turn_angle)]])

# Linearized covariance
Sigma_lin = J_td @ np.array([[sigma_angle**2]]) @ J_td.T

# Monte Carlo samples
np.random.seed(99)
noise_samples = np.random.randn(n_banana) * sigma_angle
mc_endpoints = np.array([turn_and_drive(n) for n in noise_samples])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Monte Carlo (true distribution)
ax = axes[0]
ax.scatter(mc_endpoints[:, 0], mc_endpoints[:, 1], s=3, alpha=0.2,
           color='steelblue', label='true samples')
ax.plot(0, 0, 'ko', markersize=8, label='start')
ax.plot(nominal[0], nominal[1], 'r*', markersize=14, label='nominal endpoint')
# Draw the arc that the banana lives on
arc_angles = np.linspace(turn_angle - 3*sigma_angle, turn_angle + 3*sigma_angle, 200)
arc_x = drive_dist * np.cos(arc_angles)
arc_y = drive_dist * np.sin(arc_angles)
ax.plot(arc_x, arc_y, 'k--', alpha=0.4, linewidth=1, label='arc ($r = d$)')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title(f'True Distribution ($\\sigma_\\theta = {sigma_angle_deg}°$)')
ax.legend(fontsize=9)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Right: Linearized ellipse vs true
ax2 = axes[1]
ax2.scatter(mc_endpoints[:, 0], mc_endpoints[:, 1], s=3, alpha=0.15,
            color='steelblue', label='true samples')
draw_covariance_ellipse(ax2, nominal, Sigma_lin, n_std=2,
                        fill=True, alpha=0.3, facecolor='orange',
                        edgecolor='tomato', linewidth=2.5,
                        label='Linearized $2\\sigma$')
mc_cov2 = np.cov(mc_endpoints.T)
mc_mean2 = mc_endpoints.mean(axis=0)
draw_covariance_ellipse(ax2, mc_mean2, mc_cov2, n_std=2,
                        fill=False, edgecolor='forestgreen', linewidth=2,
                        linestyle='--', label='Monte Carlo $2\\sigma$')
ax2.plot(nominal[0], nominal[1], 'r*', markersize=14)
ax2.set_xlabel('x (m)')
ax2.set_ylabel('y (m)')
ax2.set_title('Linearized (orange) vs True (blue cloud)')
ax2.legend(fontsize=9)
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Quantify the mismatch
print(f"Linearized mean:    ({nominal[0]:.3f}, {nominal[1]:.3f})")
print(f"Monte Carlo mean:   ({mc_mean2[0]:.3f}, {mc_mean2[1]:.3f})")
print(f"Mean shift:         {np.linalg.norm(nominal - mc_mean2):.3f} m")
print(f"\nLinearization misses the mean by {np.linalg.norm(nominal - mc_mean2):.3f} m")
print(f"because the true distribution is curved (banana shaped), not elliptical.")

**Key Observations:**

1. With small angle noise ($\sigma_\theta = 5°$), the linearized ellipse and
   the true distribution match well. Try it!
2. With large angle noise ($\sigma_\theta = 30°$ or more), the true
   distribution is **banana shaped**: it curves along an arc of radius $d$.
   The linearized ellipse is a poor approximation.
3. The linearized approximation also gets the **mean wrong**. The center of
   mass of the banana is pulled inward from the arc, but the linearized mean
   sits right on the nominal point.
4. When you see this kind of mismatch in practice, you should consider using
   the **Unscented Transform** or **particle filters** instead of the EKF.

---
## Exercises

Work through these exercises to solidify your understanding of uncertainty
propagation. Each one builds on the concepts from this chapter.

### Exercise 7.1: Analytical vs Numerical Jacobian

Consider the observation function that maps a landmark position $(x, y)$ to
range and bearing:

$$h(x, y) = \begin{bmatrix} \sqrt{x^2 + y^2} \\ \text{atan2}(y, x) \end{bmatrix}$$

1. Compute the Jacobian $\frac{\partial h}{\partial (x, y)}$ analytically.
2. Implement a numerical Jacobian using finite differences.
3. Compare the two at the point $(3, 4)$. They should match to several
   decimal places.

In [ ]:
# Your code here
#
# Hints:
# - The analytical Jacobian of sqrt(x^2 + y^2) w.r.t. x is x / sqrt(x^2 + y^2)
# - For the numerical Jacobian, use: dh/dx_i ≈ (h(x + eps*e_i) - h(x - eps*e_i)) / (2*eps)
# - Use eps = 1e-7 for good numerical precision
# - np.allclose() is useful for comparing the two results


### Exercise 7.2: Uncertainty Through a 2D Rotation

A point $(x, y)$ has covariance $\Sigma = \begin{bmatrix} 0.5 & 0.1 \\ 0.1 & 0.2 \end{bmatrix}$.

It is rotated by angle $\alpha$ using the rotation matrix:

$$R(\alpha) = \begin{bmatrix} \cos\alpha & -\sin\alpha \\ \sin\alpha & \cos\alpha \end{bmatrix}$$

1. What is the Jacobian of the rotation function? (This is a fun one: it is
   just $R$ itself, since rotation is linear.)
2. Propagate the covariance: $\Sigma' = R\,\Sigma\,R^T$.
3. Plot the original and rotated covariance ellipses for $\alpha = 45°$.
4. Verify: does the trace (total variance) change under rotation? Why or why not?

In [ ]:
# Your code here
#
# Hints:
# - Since rotation is a linear function, the Jacobian IS the rotation matrix.
# - Use draw_covariance_ellipse() from the helper defined above.
# - Compare np.trace(Sigma) and np.trace(Sigma_rotated).
# - Think about what rotation does geometrically to an ellipse.


### Exercise 7.3: Dead Reckoning on a Square Path

Simulate a robot driving a square with side length 4 m:

1. Drive forward 4 m (using many small steps).
2. Turn 90 degrees.
3. Repeat four times to return to the start.

At each step, propagate the covariance using the motion model from Section 7.3.
Plot the trajectory with covariance ellipses at each corner. How large is the
uncertainty when the robot returns to the start?

In [ ]:
# Your code here
#
# Hints:
# - Use the motion_model(), motion_jacobian_state(), and motion_jacobian_ctrl()
#   functions from Section 7.3.
# - For each side: drive in small steps (e.g., 20 steps of 0.2 m each).
# - For each turn: apply a single turn of pi/2 radians.
# - Store poses and covariances, then plot at the end.
# - The final uncertainty tells you why loop closure is so important!


### Exercise 7.4: Linearized vs Monte Carlo at Varying Noise Levels

Using the polar to Cartesian conversion from Section 7.2:

1. For $\sigma_\theta \in \{0.01, 0.05, 0.1, 0.2, 0.4, 0.8\}$ rad, compute
   the linearized covariance and the Monte Carlo covariance (using 5000
   samples).
2. Plot the **Frobenius norm** of the difference between the two covariance
   matrices as a function of $\sigma_\theta$.
3. At what noise level does the linearization error become significant?

In [ ]:
# Your code here
#
# Hints:
# - Frobenius norm: np.linalg.norm(A - B, 'fro')
# - Reuse the polar-to-Cartesian Jacobian from Section 7.2.
# - Use a fixed r0 and theta0, vary only sigma_theta.
# - A log-scale y-axis may be helpful for the plot.


### Exercise 7.5 (Challenge): Figure 8 Covariance Propagation

A robot drives a figure 8 path (two tangent circles). Use the following
parameterization:

- Circle radius: 3 m
- 40 steps per circle, 80 steps total
- First circle: turn left (positive $\omega$), second circle: turn right
  (negative $\omega$)

Tasks:

1. Propagate covariance through the entire path.
2. Plot the trajectory with covariance ellipses at 10 evenly spaced points.
3. Where is the uncertainty largest? At the top of a circle? At the crossing
   point? Why?
4. Compute the final covariance. Is it symmetric about any axis? Explain.

In [ ]:
# Your code here
#
# Hints:
# - For a circle of radius R with n_steps steps, use:
#     v = 2 * pi * R / n_steps  (arc length per step)
#     omega = 2 * pi / n_steps  (angle turned per step)
# - For the second circle, negate omega.
# - The crossing point is at step 40 (halfway).
# - Think about how heading uncertainty at the top of the first circle
#   affects position uncertainty for all subsequent steps.
